# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farhaan1057/FlyRank-ml-internship-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane: Refresh / Content Opportunity Scoring.

This is a ranking / scoring task, with a classification model underneath it.

- The decision it improves: which pages should an editor open and refresh first, this week?
  An editor can't review 30,000 pages, so the output has to be an ordered list, not a single number.
- Underneath the ranking sits a classification problem — is_declining_label (is this page
  losing search impressions right now?) — because "declining" is the strongest single signal
  of "needs attention," and it's an outcome I can observe (trend_direction == "down"), not
  something I have to invent.
- The final output is a score per page (probability of declining, blended with a
  transparency-first baseline rule) that gets sorted into a queue. So: classification model
  → scores → ranking.

Action this supports: a content editor opens the top of the weekly refresh queue and updates
those pages first — rewriting thin sections, fixing declining keywords, or merging near-duplicate
content. The output is directly consumable: a ranked list an editor can work top-down, today.

In [8]:
import os, subprocess

if not os.path.exists("FlyRank-ml-internship-"):
    subprocess.run(["git", "clone", "https://github.com/Farhaan1057/FlyRank-ml-internship-.git"], check=True)

os.chdir("FlyRank-ml-internship-")
print(os.getcwd())
os.listdir()

/content/FlyRank-ml-internship-/FlyRank-ml-internship-


['LICENSE',
 'notebooks',
 'skills',
 '.github',
 'CLAUDE.md',
 'data',
 '.git',
 'submission',
 'AGENTS.md',
 'docs',
 'requirements.txt',
 'outputs',
 'README.md',
 'work',
 'DATA_USE.md',
 'GUIDE.md',
 '.gitignore',
 'SETUP.md',
 'scripts']

## 2. Target or proxy

Target: is_declining_label — 1 when trend_direction == "down", else 0.

trend_direction is a defined rule, not a raw measurement: it compares impressions_last_30d
vs impressions_prev_30d and buckets the result. But the rule is applied to an observed outcome
(impressions actually falling in a later 30-day window), not something subjective. That's why
trend_direction and trend_pct can never be model features — they're built from the same data
that defines the label, so using them would let the model see the answer.

In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts(normalize=True).round(3))
df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]].head(10)

is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


,content_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1
5,content_d4084a4bc775,down,-38.9,1
6,content_9a34b442b552,down,-92.3,1
7,content_a63219c6e95a,stable,0.6,0
8,content_5e6c160719bc,down,-58.8,1
9,content_c27558df2b0c,down,-29.2,1


## 3. Success metric

Precision@50 — of the top 50 pages the model puts at the front of the refresh queue,
what fraction are actually declining (is_declining_label == 1)?

Why: editors only work through the top of the queue, never page 8,000, so a metric like AUC
that scores the whole ranked list equally is the wrong fit. The base rate is already high
(54.2% decline), so plain accuracy is easy to game. Precision@K at a K that matches real
editor capacity ties the metric directly to the decision, and it's computable today on a
baseline rule, before any model exists.

In [10]:
baseline_rank = df.sort_values(
    ["impression_tier", "days_since_last_update"], ascending=[False, False]
)
top50 = baseline_rank.head(50)
precision_at_50 = (top50["trend_direction"] == "down").mean()

print(f"rows: {len(df):,}")
print(f"base rate of decline: {(df['trend_direction'] == 'down').mean():.3f}")
print(f"naive baseline precision@50: {precision_at_50:.3f}")

rows: 30,000
base rate of decline: 0.542
naive baseline precision@50: 0.740


## 4. The unit of analysis, as a real dataframe
One row = one content item (page), aggregated over a trailing 90-day window. 30,000 rows
across 32 pseudonymized clients. content_id identifies the page; client_id is only for
grouping/joins and client-holdout splits — never a feature.

In [11]:
cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "days_since_last_update", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "trend_direction", "trend_pct",
]
df[cols].head(10)

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,0.13,44.0,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,0.03,8.5,down,-38.9
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,0.00,7.0,down,-92.3
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,0.06,21.2,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,0.09,46.0,down,-58.8
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,0.16,4.9,down,-29.2


## 5. Why ML beats a fixed rule here

I can already write a rule — flag pages with low impression_tier and old
days_since_last_update — and section 3's baseline does exactly that (precision@50 ≈ 0.74).
It's not useless, but it stalls:

- Too many signals, and they interact. word_count, content_type, main_intent,
  competition_level, avg_position, ctr, engagement_rate, ai_traffic_pct all matter together —
  a page with mediocre position but strong search_volume and rising ai_traffic_pct is a
  different opportunity than the same position with no demand. Nested if-statements get
  unreadable fast.
- Missingness is systematic, not random — keyword-context columns are blank along
  content_type lines (feedly article rows have none). A fixed rule touching those columns
  silently breaks for a chunk of the data unless someone special-cases it. A model can learn
  to weigh a feature less when it's missing.
- The baseline (0.74) is a real number to beat, not a strawman — it tells me
  impression_tier + days_since_last_update alone carry real signal. Where I'd expect it to
  break: pages with blank keyword context, or pages where high impressions mask a real
  click/engagement collapse — cases a two-column sort can't see and a model trained on the
  full 44-column feature set can.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.